In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path


In [ ]:
data_dir = Path("projects/movie_recommendation_system/data/raw/movielens")
ratings = pd.read_csv(data_dir / "ratings_small.csv")
movies = pd.read_csv(data_dir / "movies_metadata.csv")
links = pd.read_csv(data_dir / "links_small.csv")

links["tmdbId"] = pd.to_numeric(links["tmdbId"], errors="coerce").astype("Int64")
movies["id"] = pd.to_numeric(movies["id"], errors="coerce").astype("Int64")
movies = movies.dropna(subset=["id"]).drop_duplicates(subset=["id"])


In [ ]:
df = ratings.merge(links, on="movieId", how="left")
df = df.merge(
    movies[["id", "title", "genres"]],
    left_on="tmdbId",
    right_on="id",
    how="left"
)


In [ ]:
user_movie = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)


In [ ]:
user_movie.head()

## User-based recommendations for User 2

1. Find the five most similar users.
2. Collect their highly rated movies (rating >= 4.0).
3. Remove movies User 2 has already watched.
4. Average the remaining ratings for each movie.
5. Sort by average rating (then number of similar users) and show the top 20.

In [ ]:
from IPython.display import display

target_user_id = 2
n_similar_users = 5
min_rating = 4.0
n_recommendations = 20

rating_matrix = user_movie.fillna(0).to_numpy(dtype=float)
target_position = user_movie.index.get_loc(target_user_id)
denominators = (
    np.linalg.norm(rating_matrix, axis=1)
    * np.linalg.norm(rating_matrix[target_position])
)
similarity_scores = np.divide(
    rating_matrix @ rating_matrix[target_position],
    denominators,
    out=np.zeros_like(denominators),
    where=denominators != 0
)
similar_positions = np.argsort(similarity_scores)[::-1]
similar_positions = similar_positions[similar_positions != target_position]
similar_positions = similar_positions[:n_similar_users]
similar_user_ids = user_movie.index[similar_positions]

similar_users = pd.DataFrame({
    "userId": similar_user_ids,
    "similarity": similarity_scores[similar_positions]
})

neighbor_ratings = (
    user_movie.loc[similar_user_ids]
    .stack()
    .rename("rating")
    .reset_index()
)
highly_rated = neighbor_ratings[neighbor_ratings["rating"] >= min_rating]
watched_movie_ids = user_movie.columns[user_movie.loc[target_user_id].notna()]
unwatched_ratings = highly_rated[
    ~highly_rated["movieId"].isin(watched_movie_ids)
]

movie_scores = (
    unwatched_ratings.groupby("movieId", as_index=False)
    .agg(mean_rating=("rating", "mean"), similar_user_count=("rating", "count"))
)
movie_catalog = (
    df[["movieId", "title"]]
    .dropna(subset=["title"])
    .drop_duplicates("movieId")
)
recommendations = (
    movie_scores.merge(movie_catalog, on="movieId", how="left")
    .sort_values(["mean_rating", "similar_user_count"], ascending=[False, False])
    .head(n_recommendations)
    .reset_index(drop=True)
)

display(similar_users)
display(recommendations)
